Fase 2 Modul 2: Outlier, Tanggal, String

In [8]:
from pathlib import Path
import pandas as pd

# otomatis pilih yang ada: root vs subfolder
p = Path("data/dirty_tips2.csv") if Path("data/dirty_tips2.csv").exists() else Path("../data/dirty_tips2.csv")
print(f"pakai: {p}")  # harusnya ../data/dirty_tips2.csv kalau kamu di 02-python/
df = pd.read_csv(p)
print(df.shape)  # harusnya (244, 8)

pakai: ..\data\dirty_tips2.csv
(244, 8)


In [10]:
# Outlier: deteksi, jangan langsung buang

p = Path("data/dirty_tips2.csv") if Path("data/dirty_tips2.csv").exists() else Path("../data/dirty_tips2.csv")

# cara cepat: deskripsi + kuantil
print(df[["total_bill","tip"]].describe())  # max 500 langsung kelihatan
q1, q3 = df["total_bill"].quantile(0.25), df["total_bill"].quantile(0.75)
iqr = q3 - q1
batas_atas = q3 + 1.5*iqr  # aturan IQR standar
print(f"batas wajar: < {batas_atas:.2f}")
print(df[df["total_bill"] > batas_atas][["total_bill","tip","day"]])  # siapa outliernya?

# perlakuan analis (bukan hapus buta):
df["is_outlier_bill"] = df["total_bill"] > batas_atas  # flag dulu
# opsi: cap (tahan) bukan drop, biar baris tidak hilang
# df["total_bill_capped"] = df["total_bill"].clip(upper=batas_atas)

       total_bill         tip
count  244.000000  244.000000
mean    21.765492    2.970984
std     32.004375    1.472955
min      3.070000   -5.000000
25%     13.347500    2.000000
50%     17.815000    2.900000
75%     24.332500    3.562500
max    500.000000   10.000000
batas wajar: < 40.81
     total_bill    tip   day
0        500.00   1.01   sat
59        48.27   6.73   Sat
102       44.30   2.50   Sat
142       41.19   5.00  Thur
156       48.17   5.00   Sun
170       50.81  10.00   Sat
182       45.35   3.50   Sun
197       43.11   5.00  Thur
212       48.33   9.00   Sat


In [11]:
# to_datetime coba tebak format, yang gagal jadi NaT (bukan error)
df["tanggal_bersih"] = pd.to_datetime(df["tanggal"], errors="coerce", dayfirst=False)
print(df["tanggal_bersih"].isna().sum(), "gagal parse")  # "rusak" -> NaT, wajar
# ambil bagian: tahun, bulan, nama hari
df["tahun"] = df["tanggal_bersih"].dt.year
df["bulan"] = df["tanggal_bersih"].dt.month
print(df[["tanggal","tanggal_bersih"]].tail(3))

184 gagal parse
    tanggal tanggal_bersih
241   rusak            NaT
242   rusak            NaT
243   rusak            NaT


In [12]:
# String cleaning

# pola wajib: strip -> lower -> mapping -> title
df["sex_bersih"] = df["sex"].astype("string").str.strip().str.lower().map({"male":"Male","female":"Female"})
df["day_bersih"] = df["day"].astype("string").str.strip().str.title()  # sat/SAT -> Sat
print(df[["sex","sex_bersih","day","day_bersih"]].head(6))
print(df["sex_bersih"].value_counts(dropna=False))  # pastikan cuma Male/Female

       sex sex_bersih   day day_bersih
0    Male        Male   sat        Sat
1     male       Male   SAT        Sat
2   FEMALE     Female  Sat         Sat
3   Female     Female   sun        Sun
4     MALE       Male   Sun        Sun
5  female      Female   Sun        Sun
sex_bersih
Male      156
Female     88
Name: count, dtype: int64
